# Allocation Fields - Reference Values (List of Values)
All distinct valid values for allocation fields + hardcoded defaults.
Use as validation reference for the external allocation override file.

In [ ]:
%pip install openpyxl
dbutils.library.restartPython()

In [ ]:
import sys
sys.path.append("../aws_utils")
import aws_utils
from io import BytesIO
import pandas as pd

env = aws_utils.get_env_prefix()

output_path = f"/Volumes/{env}-cdh-delivery/s3_external/reports/internalreports/allocation_reference/Allocation_Reference_Values.xlsx"
print(f"Output path: {output_path}")

max_assignment_bd = spark.sql(f"SELECT max(BusinessDate) AS bd FROM `{env}-cdh`.primary.assignment").collect()[0]["bd"]
max_firmstructure_bd = spark.sql(f"SELECT max(BusinessDate) AS bd FROM `{env}-cdh`.primaryactive.firmstructure").collect()[0]["bd"]

print(f"Max assignment BusinessDate: {max_assignment_bd}")
print(f"Max firmstructure BusinessDate: {max_firmstructure_bd}")

In [ ]:
# AgentName - All Valid Values
agent_names = spark.sql(f"""
    SELECT DISTINCT AgentFullName AS AgentName
    FROM `{env}-cdh`.primary.assignment
    WHERE AgentFullName IS NOT NULL AND AgentFullName != ''
      AND BusinessDate = '{max_assignment_bd}'

    UNION

    SELECT DISTINCT EmployeeFullName AS AgentName
    FROM `{env}-cdh`.primaryactive.firmstructure
    WHERE employeeCategory = 'Employee'
      AND TeamCode != '-100'
      AND EmployeeFullName IS NOT NULL AND EmployeeFullName != ''
      AND BusinessDate = '{max_firmstructure_bd}'

    ORDER BY AgentName
""")
#display(agent_names)

In [ ]:
# TeamName - All Valid Values
team_names = spark.sql(f"""
    SELECT DISTINCT Team AS TeamName
    FROM `{env}-cdh`.primary.assignment
    WHERE Team IS NOT NULL AND Team != ''
      AND BusinessDate = '{max_assignment_bd}'

    ORDER BY TeamName
""")
#display(team_names)

In [ ]:
# ChannelName - All Valid Values
channel_names = spark.sql(f"""
    SELECT DISTINCT Channel AS ChannelName
    FROM `{env}-cdh`.primary.assignment
    WHERE Channel IS NOT NULL AND Channel != ''
      AND BusinessDate = '{max_assignment_bd}'

    ORDER BY ChannelName
""")
#display(channel_names)

In [ ]:
# ExternalCompanyName - All Valid Values
external_companies = spark.sql(f"""
    SELECT DISTINCT ExternalCompanyFullName AS ExternalCompanyName
    FROM `{env}-cdh`.primary.assignment
    WHERE ExternalCompanyFullName IS NOT NULL AND ExternalCompanyFullName != ''
      AND BusinessDate = '{max_assignment_bd}'
    ORDER BY ExternalCompanyName
""")
#display(external_companies)

In [ ]:
# AssignmentProcess - All Valid Values
assignment_processes = spark.sql(f"""
    SELECT DISTINCT AssignmentProcess
    FROM `{env}-cdh`.primary.assignment
    WHERE AssignmentProcess IS NOT NULL AND AssignmentProcess != ''
      AND BusinessDate = '{max_assignment_bd}'
    ORDER BY AssignmentProcess
""")
#display(assignment_processes)

In [ ]:
# Export to Excel (multi-sheet)
import os
os.makedirs(os.path.dirname(output_path), exist_ok=True)

output_buffer = BytesIO()
with pd.ExcelWriter(output_buffer, engine='openpyxl') as writer:
    agent_names.toPandas().to_excel(writer, sheet_name='AgentName', index=False)
    team_names.toPandas().to_excel(writer, sheet_name='TeamName', index=False)
    channel_names.toPandas().to_excel(writer, sheet_name='ChannelName', index=False)
    external_companies.toPandas().to_excel(writer, sheet_name='ExternalCompanyName', index=False)
    assignment_processes.toPandas().to_excel(writer, sheet_name='AssignmentProcess', index=False)

with open(output_path, "wb") as f:
    f.write(output_buffer.getvalue())

print(f"Reference values exported to: {output_path}")